In [0]:
# Celda 1: instalar optbinning (reinicia Python, por eso va primero)
%pip install optbinning

In [0]:
# reiniciar phyton por instalación de optbinning
dbutils.library.restartPython()

In [0]:
# Celda 2: base de modelación con las transformaciones
from pyspark.sql import functions as F

inv = spark.table("workspace.default.lc_variable_inventory").toPandas()
cand = inv.loc[inv["decision"] == "Candidata", "variable"].tolist()
cand = [c for c in cand if c not in ("funded_amnt", "funded_amnt_inv", "fico_range_high",
                                     "earliest_cr_line", "emp_length")]
cand = [c for c in cand if c not in ("funded_amnt", "funded_amnt_inv", "fico_range_high",
                                     "earliest_cr_line", "emp_length", "addr_state")]
cand += ["credit_hist_months", "emp_length_yrs"]

mod = (spark.table("workspace.default.lc_population")
       # Antigüedad crediticia en meses al momento de originar
       .withColumn("credit_hist_months",
                   F.months_between("issue_date", F.to_date("earliest_cr_line", "MMM-yyyy")).cast("int"))
       # "< 1 year" -> 0, "10+ years" -> 10, "n/a" -> nulo
       .withColumn("emp_length_yrs",
                   F.when(F.col("emp_length").startswith("<"), 0)
                    .otherwise(F.regexp_extract("emp_length", r"(\d+)", 1).cast("int")))
       .select("id", "issue_year", "sample", "target", *cand))

mod.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.lc_model_base")
print(f"Variables candidatas: {len(cand)}")


In [0]:
# Celda 3: binning óptimo e IV sobre DEV
from optbinning import BinningProcess

dev_pd = spark.table("workspace.default.lc_model_base").filter("sample = 'DEV'").toPandas()
X, y = dev_pd[cand], dev_pd["target"].values
categoricas = [c for c in cand if X[c].dtype == "object"]   # purpose, home_ownership, etc.

bp = BinningProcess(variable_names=cand, categorical_variables=categoricas)
bp.fit(X, y)

resumen = bp.summary().sort_values("iv", ascending=False)
display(resumen[["name", "dtype", "n_bins", "iv", "gini"]])


In [0]:
# Celda 4: correlación entre WOE y selección con umbral 0,7 y 0,6
import pandas as pd
import numpy as np

# Variables con IV >= 0,02 y más de un grupo, ordenadas por IV descendente
sel_iv = (resumen.loc[(resumen["iv"] >= 0.02) & (resumen["n_bins"] > 1), ["name", "iv"]]
                 .rename(columns={"name": "variable"})
                 .reset_index(drop=True))
sel_iv["iv"] = sel_iv["iv"].astype(float)
vars_iv = sel_iv["variable"].tolist()
print(f"Variables con IV >= 0,02: {len(vars_iv)}")

# Transformar DEV a WOE y calcular la correlación (Pearson sobre WOE)
X_woe = bp.transform(X, metric="woe")[vars_iv]
corr = X_woe.corr()

def seleccion_por_correlacion(umbral):
    seleccionadas, filas = [], []
    for v in vars_iv:
        choques = [(s, corr.loc[v, s]) for s in seleccionadas if abs(corr.loc[v, s]) > umbral]
        if choques:
            s, r = max(choques, key=lambda t: abs(t[1]))   # el choque más fuerte
            filas.append((v, "Excluida", s, round(r, 3)))
        else:
            seleccionadas.append(v)
            filas.append((v, "Seleccionada", None, None))
    return pd.DataFrame(filas, columns=["variable", "decision", "choca_con", "r"])

r07 = seleccion_por_correlacion(0.7)
r06 = seleccion_por_correlacion(0.6)

comparacion = (sel_iv.merge(r07, on="variable")
                     .merge(r06, on="variable", suffixes=("_0.7", "_0.6")))
comparacion["iv"] = comparacion["iv"].round(4)
display(comparacion)

print(f"Seleccionadas con 0,7: {(r07['decision'] == 'Seleccionada').sum()} | "
      f"con 0,6: {(r06['decision'] == 'Seleccionada').sum()}")



In [0]:
# Celda 5: pares de variables con |r| > 0,6, para ver los grupos que se repiten
pares = (corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))   # solo la mitad superior
             .stack().reset_index())
pares.columns = ["var_1", "var_2", "r"]
display(pares[pares["r"].abs() > 0.6].sort_values("r", key=abs, ascending=False))


In [0]:
# Celda 6: guardar selección, tabla WOE y binning
import pickle

vars_07 = r07.loc[r07["decision"] == "Seleccionada", "variable"].tolist()
vars_06 = r06.loc[r06["decision"] == "Seleccionada", "variable"].tolist()

# 1. Comparación de umbrales (documento de validación)
comparacion.columns = [c.replace(".", "") for c in comparacion.columns]   # Delta no acepta puntos
spark.createDataFrame(comparacion).write.mode("overwrite").option("overwriteSchema", "true") \
     .saveAsTable("workspace.default.lc_correlation_selection")

# 2. Listas de variables por modelo
listas = pd.DataFrame({"variable": vars_07})
listas["modelo_a_07"] = True
listas["modelo_b_06"] = listas["variable"].isin(vars_06)
spark.createDataFrame(listas).write.mode("overwrite").option("overwriteSchema", "true") \
     .saveAsTable("workspace.default.lc_model_variables")

# 3. Tabla WOE para DEV y OOT (las 16 variables; el modelo B usa un subconjunto)
full_pd = spark.table("workspace.default.lc_model_base").toPandas()
woe = bp.transform(full_pd[cand], metric="woe")[vars_07]
woe.columns = [f"woe_{c}" for c in vars_07]
lc_woe = pd.concat([full_pd[["id", "issue_year", "sample", "target"]].reset_index(drop=True),
                    woe.reset_index(drop=True)], axis=1)
spark.createDataFrame(lc_woe).write.mode("overwrite").option("overwriteSchema", "true") \
     .saveAsTable("workspace.default.lc_woe")

# 4. Objeto de binning en un volumen de artefactos
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.artifacts")
with open("/Volumes/workspace/default/artifacts/binning_process.pkl", "wb") as f:
    pickle.dump(bp, f)

print(f"Modelo A: {len(vars_07)} variables | Modelo B: {len(vars_06)} variables")
print(f"Tabla WOE: {len(lc_woe):,} filas")
